# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shile/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from google.colab import userdata
from huggingface_hub import login
import duckdb
import pandas as pd


In [4]:
login(userdata.get('flyrank_huggingface'))

In [36]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2025-01/data_0.parquet' (HTTP 401)

In [32]:
rel = "hf://datasets/FlyRank/internship-warehouse"
query = f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 100000 OFFSET 0"

df = con.sql(query).df()

df.head()

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2025-01/data_0.parquet' (HTTP 401)

In [5]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('flyrank_huggingface')}')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

query = f'SELECT * FROM read_parquet("{rel}/fact_content_daily_performance/**/*.parquet") LIMIT 100000 OFFSET 0'

df = con.sql(query).df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [21]:
df.shape

(100000, 31)

In [7]:
df['report_date'].unique()

<DatetimeArray>
['2025-01-27 00:00:00', '2025-01-28 00:00:00', '2025-01-29 00:00:00',
 '2025-01-30 00:00:00', '2025-01-31 00:00:00', '2025-02-10 00:00:00',
 '2025-02-11 00:00:00', '2025-02-12 00:00:00', '2025-02-13 00:00:00',
 '2025-02-14 00:00:00', '2025-02-20 00:00:00', '2025-02-21 00:00:00',
 '2025-02-22 00:00:00', '2025-02-01 00:00:00', '2025-02-02 00:00:00',
 '2025-02-03 00:00:00', '2025-02-04 00:00:00', '2025-02-05 00:00:00',
 '2025-02-06 00:00:00', '2025-02-07 00:00:00', '2025-02-08 00:00:00',
 '2025-02-09 00:00:00', '2025-02-23 00:00:00', '2025-02-24 00:00:00',
 '2025-02-25 00:00:00', '2025-02-26 00:00:00', '2025-02-27 00:00:00',
 '2025-02-28 00:00:00', '2025-02-15 00:00:00', '2025-02-16 00:00:00',
 '2025-02-17 00:00:00', '2025-02-18 00:00:00', '2025-02-19 00:00:00',
 '2025-03-01 00:00:00', '2025-03-02 00:00:00', '2025-03-03 00:00:00',
 '2025-03-19 00:00:00', '2025-03-20 00:00:00', '2025-03-21 00:00:00',
 '2025-03-15 00:00:00', '2025-03-16 00:00:00']
Length: 41, dtype: datetime

In [8]:
snapshot_df = df[df['report_date'] == '2025-03-16 00:00:00'].copy()

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
snapshot_df.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude',
       'ai_meta', 'ai_other', 'scroll_events', 'month'],
      dtype='object')

In [ ]:
context = ['client_hash_id', 'content_hash_id', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
feature = ['sessions_ai', 'sessions_social', 'sessions_referral', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', ]

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.